# 5.5 - Structured Output & Tool Use

**Module 5 - Prompt Engineering** | Netsetos GenAI Engineering

This notebook turns the model from something that *talks* into something that *plugs into code*. You climb the reliability ladder to schema-constrained JSON with a Pydantic model, add a boundary validator so the values are true and not just well-shaped, then hand the model your Python functions as tools so it can act and fetch live data - one tool, several tools in parallel, and finally both patterns combined into a small product API. Every technique runs against real Gemini API calls through the unified `google-genai` SDK, but the same shapes are native on OpenAI and Claude.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - seed the run for reproducibility

A one-line marker cell. The lesson leans on deterministic, repeatable outputs so that the printed results below every cell actually match what you see when you run them. There is nothing to execute here beyond the comment itself.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

This is a placeholder/marker cell, not a computation. It documents that the notebook's examples are meant to be reproducible - the real environment prep happens in the next cell.

**How the code works - step by step**
- **The comment line** - flags that the notebook intends stable, repeatable output; no variable is set and nothing runs.

**In one line:** a reproducibility note, not code that does work.

**What you'll see:** (no output - it is a comment-only marker cell)

## Setup - one client and one helper we reuse all lesson

We build the Gemini client once and wrap it in a tiny `generate()` helper so every later example shares the same error-handled call. Everything special about structured output and tool use will be passed later through the `config` argument - so this wrapper is the single door all six concepts walk through.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# pip install "google-genai>=1.0.0" pydantic
from google import genai
from google.genai import types
from pydantic import BaseModel, Field
import os, json

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-3-flash"

def generate(prompt: str, config: types.GenerateContentConfig = None):
    """Thin wrapper so every example shares one error-handled call."""
    try:
        return client.models.generate_content(model=MODEL, contents=prompt, config=config)
    except Exception as e:
        raise RuntimeError(f"API call failed: {e}")

print(generate("Reply with one word: ready").text)
# Output: ready

**Explanation**

The environment-prep cell: install, import, build a reusable client, and define a thin wrapper that turns any network failure into one clear error instead of a mid-pipeline crash. Read it as the shared plumbing - the interesting arguments (`config`) arrive in later cells.

**How the code works - step by step**
- **`pip install` + imports** - `google-genai` (the current unified SDK; the old `google.generativeai` was deprecated in 2025), `pydantic` for schemas, plus `os`/`json`.
- **`genai.Client(api_key=...)`** - one reusable client, replacing the old global `configure()` + per-call `GenerativeModel` pattern.
- **`MODEL = "gemini-3-flash"`** - the model every call targets.
- **`generate(prompt, config)`** - wraps `client.models.generate_content` in a try/except that re-raises as a single `RuntimeError`, so every example inherits the same error handling.
- **The test call** - `generate("Reply with one word: ready")` confirms the key and client work end to end.

**In one line:** build the client once, wrap it once, prove it works.

**What you'll see:** Prints `ready` - the model's one-word confirmation that the client and API key are wired up.

## 1 - Schema-constrained output: guaranteed JSON

Instead of asking nicely for JSON, you hand the model a Pydantic model as a schema and get back a validated object every time. This is the top rung of the reliability ladder: constrained decoding blocks any token that would break the schema, so the output always parses and always has your fields and types.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
class Product(BaseModel):
    name: str
    rating: int = Field(ge=1, le=5)
    in_stock: bool
    tags: list[str]

resp = generate(
    "Extract a product record from: 'The Buds 3 are great, 4 stars, in stock. #audio #anc'",
    types.GenerateContentConfig(
        response_mime_type="application/json",   # JSON mode...
        response_schema=Product,                  # ...constrained to THIS schema
    ),
)
product = resp.parsed          # already a validated Product instance
print(product.name, product.rating, product.tags)
# Output: Buds 3 4 ['audio', 'anc']

**Explanation**

The first real payoff cell: a Pydantic class doubles as the output contract. Passing it as `response_schema` switches on constrained decoding, and `resp.parsed` returns a ready-made typed object with no `json.loads` and no try/except around parsing. The core idea - the model IS being forced to fill your form, not to write prose you clean up.

**How the code works - step by step**
- **`class Product(BaseModel)`** - declares the exact shape: `name` (str), `rating` (int constrained to 1-5 via `Field(ge=1, le=5)`), `in_stock` (bool), `tags` (list of str). The SDK converts this to a JSON Schema for you.
- **`response_mime_type="application/json"`** - turns on JSON mode (valid JSON guaranteed).
- **`response_schema=Product`** - narrows that further to constrained decoding against *this* schema, so an int field can never receive a string.
- **`resp.parsed`** - hands back an already-validated `Product` instance; no manual parsing because the output is guaranteed to match.

**In one line:** the Pydantic model is the contract - pass it in, get a typed object out.

**What you'll see:** Prints `Buds 3 4 ['audio', 'anc']` - the extracted name, the 1-5 rating, and the parsed tag list, straight off a typed `Product`.

## 2 - Validate at the boundary: syntax vs semantics

Constrained decoding guarantees the *shape*, never the *truth* - a `rating` will always be 1-5 but could still be the wrong number. This cell adds a Pydantic field validator that checks meaning (does this GSTIN actually look like a GSTIN?) and wraps the whole extraction in a bounded retry loop.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
from pydantic import field_validator, ValidationError

class Invoice(BaseModel):
    gstin: str
    amount: float = Field(gt=0)

    @field_validator("gstin")
    @classmethod
    def check_gstin(cls, v: str) -> str:
        if not (len(v) == 15 and v[:2].isdigit()):   # real check is stricter
            raise ValueError("not a valid GSTIN")
        return v

def extract_invoice(text: str, retries: int = 2) -> Invoice:
    for attempt in range(retries + 1):
        resp = generate(f"Extract invoice fields from: {text}",
            types.GenerateContentConfig(response_mime_type="application/json", response_schema=Invoice))
        try:
            # parse the raw JSON ourselves so the validators run and RAISE
            return Invoice.model_validate_json(resp.text)
        except ValidationError as e:
            if attempt == retries: raise
            # else: loop; a stronger version appends str(e) to the prompt

inv = extract_invoice("GSTIN 29ABCDE1234F1Z5, total 4500.00")
print(inv.gstin, inv.amount)
# Output: 29ABCDE1234F1Z5 4500.0

**Explanation**

A production-hardening cell layered on top of Concept 1. The schema still handles syntax; a `field_validator` handles semantics that no type constraint can express, and it must be made to *raise* so the retry can fire. Read it as: guaranteed shape plus validated meaning plus a bounded number of second chances.

**How the code works - step by step**
- **`class Invoice`** with `@field_validator("gstin")** - beyond typing `gstin: str`, the validator checks length is 15 and the first two chars are digits, raising `ValueError` otherwise (the real GSTIN check is stricter).
- **`Invoice.model_validate_json(resp.text)`** - deliberately parses the raw JSON *ourselves* rather than using `resp.parsed`: `resp.parsed` swallows a validator failure and returns `None`, so the retry would never fire; `model_validate_json` runs the validators and *raises* `ValidationError`.
- **The `for attempt in range(retries + 1)` loop** - re-runs the prompt on a semantic miss (sampling is non-deterministic, so a re-run can return a different valid value), re-raises after the last attempt; a stronger version appends `str(e)` so the model self-corrects.
- **The India-format point** - GSTIN/PAN/Aadhaar are the classic trap: a plausible string that fails a checksum, which only a boundary validator catches.

**In one line:** schema checks the boxes, the validator checks the answers, the loop gives bounded retries.

**What you'll see:** Prints `29ABCDE1234F1Z5 4500.0` - the extracted GSTIN passed the validator and the amount parsed as a float, so no retry was needed.

## 3 - Tool use: let the model call your functions

Here the model stops answering directly and instead proposes a *function call*. You pass a plain Python function as a tool; the SDK reads its signature and docstring to build the schema, and Automatic Function Calling runs the function and feeds the result back for a final natural-language answer.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
def get_stock(sku: str) -> int:
    """Return the number of units in stock for a product SKU."""
    return {"buds-3": 42, "case-x": 0}.get(sku, 0)

# Pass the Python function as a tool. The SDK reads the signature + docstring
# to build the schema, and Automatic Function Calling runs it for you.
resp = generate(
    "How many buds-3 do we have in stock?",
    types.GenerateContentConfig(tools=[get_stock]),
)
print(resp.text)
# Output: You have 42 units of buds-3 in stock.

**Explanation**

The pivot cell of the lesson: a tool is just your function with a docstring. This turns the model from a chatbot into something that acts - it decides which function to call and with what typed arguments, and your code (via AFC) executes it. The dispatcher proposes; the field crew disposes.

**How the code works - step by step**
- **`get_stock(sku: str) -> int`** - an ordinary function; its type hints and docstring become the tool's JSON Schema automatically.
- **`tools=[get_stock]`** in the config - registers the function as a callable tool.
- **The model emits a call, not text** - internally it returns `get_stock(sku="buds-3")` with structured, typed arguments.
- **Automatic Function Calling** - the SDK runs the function (result `42`), feeds it back to the model, and continues to the final answer, all inside the one `generate` call. (Disable AFC to run the loop yourself - the ReAct pattern from Lesson 5.3.)

**In one line:** your function becomes a schema, the model calls it, AFC closes the loop.

**What you'll see:** Prints `You have 42 units of buds-3 in stock.` - the model called `get_stock`, got 42, and wrote the answer from the real result.

## 4 - Many tools, parallel calls, and routing

Give the model a toolbox instead of a single tool and it becomes a router: it reads each docstring, picks the right tool(s), and can call several at once. For a question that needs both stock and price, it fires both calls in one turn.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
def get_stock(sku: str) -> int:
    """Units in stock for a SKU."""
    return {"buds-3": 42}.get(sku, 0)

def get_price(sku: str) -> int:
    """Price in rupees for a SKU."""
    return {"buds-3": 3299}.get(sku, 0)

resp = generate(
    "How many buds-3 are in stock and what do they cost?",
    types.GenerateContentConfig(tools=[get_stock, get_price]),   # model picks and routes
)
print(resp.text)
# Output: There are 42 buds-3 in stock at 3299 rupees each.
# (the model called BOTH tools in parallel, in one turn)

**Explanation**

An extension of Concept 3 to multiple tools. The new idea is routing plus parallelism: with a list of tools the model chooses which to invoke, and a multi-part request collapses into a single round-trip because both calls happen at once rather than in sequence. More tools means more routing decisions, so crisp non-overlapping docstrings matter.

**How the code works - step by step**
- **`get_stock` and `get_price`** - two small functions, each with a one-line docstring the model uses to route.
- **`tools=[get_stock, get_price]`** - passes both; the model would call only `get_price` for a price-only question.
- **Parallel calls** - for "stock and price" the model emits both calls in one turn; the SDK runs them (you can execute concurrently) and returns both results together - one extra turn, not two.
- **Forced choice** - a `tool_config` can require the model to call a specific tool (or any tool) when an action is mandatory.

**In one line:** hand over a toolbox, the model routes and calls in parallel, collapsing round-trips.

**What you'll see:** Prints `There are 42 buds-3 in stock at 3299 rupees each.` - the model called both tools in the same turn and merged the two results into one answer.

## 5 - Cross-provider and production: extraction + tool use together

The final cell combines both halves of the lesson into a small product endpoint: one function does pure schema-constrained extraction to build a validated record, the other does pure tool use to answer a live question. The same two config surfaces exist on OpenAI and Claude, so the schema and the tool function are reused as-is across providers.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`).

In [ ]:
# Project: a product endpoint that extracts a validated record AND can
# answer live stock questions via a tool - structured output + tool use together.
class Product(BaseModel):
    name: str
    rating: int = Field(ge=1, le=5)
    in_stock: bool

def stock_lookup(sku: str) -> int:
    """Live units in stock for a SKU."""
    return {"buds-3": 42}.get(sku, 0)

def extract_product(text: str) -> Product:
    return generate(f"Extract a product record from: {text}",
        types.GenerateContentConfig(response_mime_type="application/json",
                                    response_schema=Product)).parsed

def answer_query(question: str) -> str:
    return generate(question, types.GenerateContentConfig(tools=[stock_lookup])).text

print(extract_product("Buds 3, 4 stars, in stock"))
# Output: name='Buds 3' rating=4 in_stock=True
print(answer_query("Is buds-3 in stock right now?"))
# Output: Yes - there are 42 units of buds-3 in stock.

**Explanation**

The capstone cell that shows the two patterns side by side, which is the shape of real product APIs: structured extraction to populate a database, tool calling to answer live questions against it. The point is separation - `extract_product` never calls a tool, `answer_query` never validates a schema, and swapping providers only touches the config, not the Pydantic model or the function.

**How the code works - step by step**
- **`class Product`** - the same schema pattern from Concept 1, reused unchanged.
- **`stock_lookup(sku)`** - a live tool function, same pattern as Concept 3.
- **`extract_product(text)`** - pure structured output: free text in, a validated `Product` out via `response_schema`, `.parsed` returned directly - no parsing code.
- **`answer_query(question)`** - pure tool use: the model routes the question to `stock_lookup` and answers from the real number.
- **The anti-pattern warning** - shipping `json.loads(resp.text)` on a "Return ONLY JSON" prompt passes every test then crashes when the model adds a greeting or a markdown fence; the fix is the whole lesson - schema-constrain, validate, and let tools fetch what the model cannot know.

**In one line:** extraction fills your data, tool use answers against it - one schema, one tool, portable across providers.

**What you'll see:** Prints two lines: `name='Buds 3' rating=4 in_stock=True` from the extraction, then `Yes - there are 42 units of buds-3 in stock.` from the tool call.

You climbed from "trust me, it's JSON" to a signed contract: `response_schema` guarantees the *shape* every time, a Pydantic `field_validator` at the boundary guarantees the *meaning*, and passing your functions as `tools` lets the model propose typed calls that your code executes - one at a time or several in parallel. The final project shows the production combination: schema-constrained extraction to populate your data, tool calling to answer live questions against it. From here, tools grow into a full ecosystem and MCP in Module 10, chain into agents in Module 11, and retrieval becomes just another tool the model calls in Module 8.